# SIM Intervention Demand Forecasting — Model Comparison and Selection

## 6.1. Objective

The objective of this stage is to evaluate whether a simpler and more interpretable forecasting model can achieve performance comparable to the previously developed machine learning models.

The initial modeling approach evaluated LightGBM and CatBoost, which achieved modest reductions in MAE compared with the historical baseline but showed greater negative prediction bias.

We will introduce Linear Regression as an additional candidate and evaluate it using the same 48 monthly chronological validation windows from 2021 through 2024.

The comparison will consider:

- Mean Absolute Error (MAE).
- Prediction bias.
- Performance across operational divisions.
- Performance across validation years.
- Model interpretability and complexity.

The goal is to select the simplest forecasting approach that provides adequate predictive performance.

The reserved 2025–2026 final test period will remain excluded from model selection.

## 6.2. Import Libraries and Load Datasets

We will load the modeling dataset, the chronological validation schedule, and the predictions from the previously evaluated forecasting methods.

We will use scikit-learn to implement Linear Regression. A preprocessing pipeline will handle categorical variables and missing numerical values using only the training data available in each validation window.

The existing predictions will allow us to compare all methods without retraining LightGBM or CatBoost.

In [ ]:

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Load modeling data
df_modeling = pd.read_csv(
    "../../data/interim/modelling/SIM_Features_Modeling.csv",
    parse_dates=["Date"]
)

# Load validation schedule
df_schedule = pd.read_csv(
    "../../data/interim/modelling/SIM_Chronological_Validation_Schedule.csv",
    parse_dates=[
        "Validation_Start",
        "Validation_End",
        "Training_Cutoff"
    ]
)

# Load previously evaluated model predictions
df_previous_predictions = pd.read_csv(
    "../../data/interim/modelling/SIM_Model_Validation_Predictions.csv",
    parse_dates=["Date"]
)

# Sort observations by date and division
df_modeling = df_modeling.sort_values(
    ["Date", "DIVISION"]
).reset_index(drop=True)

# Verify the datasets
print("Modeling observations:", len(df_modeling))
print("Validation months:", len(df_schedule))
print("Previous validation predictions:", len(df_previous_predictions))

print("\nPrevious prediction columns:")
print(df_previous_predictions.columns.tolist())


Modeling observations: 14748
Validation months: 48
Previous validation predictions: 8760

Previous prediction columns:
['Date', 'DIVISION', 'Actual_Count', 'Baseline_Prediction', 'LightGBM_Prediction', 'CatBoost_Prediction']


## 6.3. Define Features and Preprocessing Pipeline

We will prepare the 12 candidate predictive features for Linear Regression.

The `DIVISION` variable will be encoded using One-Hot Encoding because division identifiers represent categories rather than continuous numerical values.

Missing numerical feature values will be replaced using the median calculated from the training observations.

A scikit-learn preprocessing pipeline will ensure that these transformations are fitted only on the training data for each chronological validation window.

This approach will allow Linear Regression to use the same candidate predictors as LightGBM and CatBoost while preventing preprocessing-related data leakage.

In [ ]:

# Define categorical features
categorical_features = ["DIVISION"]

# Define numerical features
numerical_features = [
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend",
    "DayOfYear",
    "Lag_3",
    "Lag_7",
    "Lag_14",
    "Lag_28",
    "Rolling_Mean_7",
    "Rolling_Mean_14"
]

# Combine all predictive features
feature_columns = categorical_features + numerical_features

# Create numerical preprocessing
numerical_preprocessing = SimpleImputer(
    strategy="median"
)

# Create categorical preprocessing
categorical_preprocessing = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine preprocessing operations
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_preprocessing, numerical_features),
        ("categorical", categorical_preprocessing, categorical_features)
    ]
)

# Create the Linear Regression pipeline
linear_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("regression", LinearRegression())
    ]
)

# Verify feature configuration
print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))
print("Total predictive features:", len(feature_columns))

print("\nModel pipeline:")
print(linear_model)


Categorical features: 1
Numerical features: 11
Total predictive features: 12

Model pipeline:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numerical',
                                                  SimpleImputer(strategy='median'),
                                                  ['DayOfWeek', 'Month',
                                                   'DayOfMonth', 'IsWeekend',
                                                   'DayOfYear', 'Lag_3',
                                                   'Lag_7', 'Lag_14', 'Lag_28',
                                                   'Rolling_Mean_7',
                                                   'Rolling_Mean_14']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['DIVISION'])])),
                ('regression', LinearRegression())])


## 6.4. Train the First Linear Regression Model

We will train an initial Linear Regression model using historical observations available through December 29, 2020.

The model will be evaluated on January 2021, using the same chronological training and validation periods as the initial LightGBM and CatBoost models.

All preprocessing operations will be fitted using training observations only.

We will first verify that the model generates predictions successfully before calculating its forecasting accuracy.

In [ ]:

# Select the first validation window
first_window = df_schedule.iloc[0]

# Define the training and validation periods
training_cutoff = first_window["Training_Cutoff"]
validation_start = first_window["Validation_Start"]
validation_end = first_window["Validation_End"]

# Select training observations
df_train = df_modeling[
    df_modeling["Date"] <= training_cutoff
].copy()

# Select validation observations
df_val = df_modeling[
    (df_modeling["Date"] >= validation_start) &
    (df_modeling["Date"] <= validation_end)
].copy()

# Prepare input features and target
X_train = df_train[feature_columns]
y_train = df_train["Intervention_Count"]

X_val = df_val[feature_columns]

# Train Linear Regression
linear_model.fit(X_train, y_train)

# Generate predictions
linear_predictions = linear_model.predict(X_val)

# Store the results
df_linear_first = df_val[
    ["Date", "DIVISION", "Intervention_Count"]
].copy()

df_linear_first["Predicted_Count"] = linear_predictions

# Display results
print("Linear Regression training completed.")
print("Training observations:", len(df_train))
print("Validation observations:", len(df_val))
print("Number of predictions:", len(linear_predictions))
print("Missing predictions:", np.isnan(linear_predictions).sum())

display(df_linear_first.head(10))


Linear Regression training completed.
Training observations: 2184
Validation observations: 186
Number of predictions: 186
Missing predictions: 0


,Date,DIVISION,Intervention_Count,Predicted_Count
2196,2021-01-01,1,18.0,26.262695
2197,2021-01-01,2,41.0,52.819793
2198,2021-01-01,3,48.0,51.549891
2199,2021-01-01,4,53.0,56.753947
2200,2021-01-01,5,42.0,55.034881
2201,2021-01-01,6,27.0,27.083542
2202,2021-01-02,1,22.0,26.406278
2203,2021-01-02,2,35.0,49.296900
2204,2021-01-02,3,48.0,49.323978
2205,2021-01-02,4,48.0,58.307746


## 6.5. Evaluate the First Linear Regression Model

We will evaluate the initial Linear Regression model using the 186 validation observations from January 2021.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors. Prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

We will also calculate MAE by operational division to examine differences in forecasting performance.

These initial results will be compared with LightGBM and CatBoost for the same month. A complete 48-month validation will be required before drawing conclusions about overall model performance.

In [ ]:

# Calculate prediction errors
df_linear_first["Error"] = (
    df_linear_first["Predicted_Count"] -
    df_linear_first["Intervention_Count"]
)

# Calculate overall MAE and Bias
linear_mae = df_linear_first["Error"].abs().mean()
linear_bias = df_linear_first["Error"].mean()

print("Linear Regression Validation Results (January 2021)")
print("Total predictions:", len(df_linear_first))
print("MAE:", round(linear_mae, 3))
print("Bias:", round(linear_bias, 3))

# Calculate MAE by division
linear_division_mae = (
    df_linear_first.groupby("DIVISION")["Error"]
    .apply(lambda errors: errors.abs().mean())
)

print("\nMAE by division:")
display(linear_division_mae.round(3))


Linear Regression Validation Results (January 2021)
Total predictions: 186
MAE: 6.918
Bias: 0.639

MAE by division:


DIVISION
1     5.494
2     7.149
3     5.610
4     7.671
5    11.429
6     4.157
Name: Error, dtype: float64

**Methodological Observation**

The initial Linear Regression model was evaluated using 186 division-day observations from January 2021, with the same chronological training and validation periods used for LightGBM and CatBoost.

Linear Regression achieved an MAE of 6.918 interventions and a positive prediction bias of 0.639.

For this month, Linear Regression achieved a lower MAE and smaller absolute bias than both LightGBM and CatBoost. It also obtained the lowest MAE in five of the six operational divisions.

These initial findings suggest that a simpler linear model may provide competitive forecasting performance. However, the complete 48-month chronological validation is required before drawing conclusions about overall accuracy and model selection.

## 6.6. Train Linear Regression Across Monthly Validation Windows

We will evaluate Linear Regression using the same 48 monthly chronological validation windows applied to LightGBM and CatBoost.

For each validation month, a new Linear Regression pipeline will be fitted using historical observations available through the corresponding D−3 training cutoff.

The preprocessing pipeline will calculate missing-value replacements and categorical encodings using training data only.

The model will remain fixed during each validation month, while daily predictive features may use newly available historical intervention counts through D−3.

All validation predictions will be stored for subsequent comparison with the baseline and previously evaluated machine learning models. The reserved 2025–2026 final test period will remain excluded.

In [ ]:

from sklearn.base import clone

# Store predictions from all validation months
all_linear_predictions = []

# Process each monthly validation window
for _, window in df_schedule.iterrows():

    # Select historical training observations
    monthly_train = df_modeling[
        df_modeling["Date"] <= window["Training_Cutoff"]
    ]

    # Select validation observations
    monthly_val = df_modeling[
        (df_modeling["Date"] >= window["Validation_Start"]) &
        (df_modeling["Date"] <= window["Validation_End"])
    ]

    # Separate features and target
    X_train_month = monthly_train[feature_columns]
    y_train_month = monthly_train["Intervention_Count"]

    X_val_month = monthly_val[feature_columns]

    # Create a fresh copy of the full pipeline
    monthly_linear_model = clone(linear_model)

    # Fit preprocessing and regression using training data only
    monthly_linear_model.fit(X_train_month, y_train_month)

    # Generate predictions for the validation month
    monthly_predictions = monthly_linear_model.predict(X_val_month)

    # Store the predictions and actual counts
    monthly_results = monthly_val[
        ["Date", "DIVISION", "Intervention_Count"]
    ].copy()

    monthly_results["Linear_Prediction"] = monthly_predictions

    all_linear_predictions.append(monthly_results)

# Combine all monthly predictions
df_linear_predictions = pd.concat(
    all_linear_predictions,
    ignore_index=True
)

# Verify results
print("Validation months processed:", len(all_linear_predictions))
print("Total predictions:", len(df_linear_predictions))

print(
    "Missing predictions:",
    df_linear_predictions["Linear_Prediction"].isna().sum()
)

print("\nFirst five predictions:")
display(df_linear_predictions.head())


Validation months processed: 48
Total predictions: 8760
Missing predictions: 0

First five predictions:


,Date,DIVISION,Intervention_Count,Linear_Prediction
0,2021-01-01,1,18.0,26.262695
1,2021-01-01,2,41.0,52.819793
2,2021-01-01,3,48.0,51.549891
3,2021-01-01,4,53.0,56.753947
4,2021-01-01,5,42.0,55.034881


## 6.7. Evaluate Linear Regression Validation Performance

We will evaluate Linear Regression predictions across the 48 monthly chronological validation windows covering 2021 through 2024.

Mean Absolute Error (MAE) will measure the average magnitude of forecasting errors, while prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

We will also evaluate performance separately by operational division and calendar year to examine differences in forecasting accuracy and temporal stability.

These results will provide the evidence needed to assess whether Linear Regression is a competitive alternative to LightGBM, CatBoost, and the historical baseline.

In [ ]:

# Calculate prediction errors
df_linear_predictions["Error"] = (
    df_linear_predictions["Linear_Prediction"] -
    df_linear_predictions["Intervention_Count"]
)

# Calculate overall MAE and Bias
linear_overall_mae = (
    df_linear_predictions["Error"].abs().mean()
)

linear_overall_bias = (
    df_linear_predictions["Error"].mean()
)

# Display overall results
print("Linear Regression Validation Results (2021-2024)")
print("Total predictions:", len(df_linear_predictions))
print("Overall MAE:", round(linear_overall_mae, 3))
print("Overall Bias:", round(linear_overall_bias, 3))

# Calculate performance by division
linear_division_metrics = (
    df_linear_predictions.groupby("DIVISION")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by division:")
display(linear_division_metrics.round(3))

# Extract the forecast year
df_linear_predictions["Year"] = (
    df_linear_predictions["Date"].dt.year
)

# Calculate performance by year
linear_yearly_metrics = (
    df_linear_predictions.groupby("Year")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by year:")
display(linear_yearly_metrics.round(3))


Linear Regression Validation Results (2021-2024)
Total predictions: 8760
Overall MAE: 8.384
Overall Bias: -1.34

Performance by division:


,MAE,Bias
DIVISION,,
1,6.413,-1.079
2,8.443,-1.251
3,9.716,-1.521
4,8.780,-1.200
5,9.150,-1.255
6,7.805,-1.732



Performance by year:


,MAE,Bias
Year,,
2021,7.292,-0.928
2022,7.568,-0.965
2023,10.619,-1.520
2024,8.059,-1.945


**Methodological Observation**

Linear Regression was evaluated across 48 monthly chronological validation windows covering 8,760 division-day observations from 2021 through 2024.

The model achieved an overall MAE of 8.384 interventions and a prediction bias of −1.340, indicating a tendency to underestimate daily intervention counts.

Linear Regression achieved a slightly lower MAE than the historical baseline (8.480), representing an improvement of approximately 1.13%. However, LightGBM (8.150) and CatBoost (8.127) achieved greater reductions in absolute forecasting error.

Linear Regression demonstrated less systematic underestimation than the two gradient boosting models, although the baseline remained closest to zero bias.

Annual results showed that Linear Regression did not consistently outperform the baseline in every year. The largest performance gap relative to the advanced models occurred in 2023.

These findings indicate that Linear Regression offers a simpler, competitive alternative, but does not achieve the lowest overall MAE. Further comparison is required before selecting the preferred forecasting approach.

## 6.8. Match Predictions from All Four Methods

We will match Linear Regression predictions with the previously evaluated baseline, LightGBM, and CatBoost predictions.

The datasets will be joined using forecast date and operational division.

We will verify that all 8,760 observations are matched, that date-division combinations are unique, and that the actual intervention counts agree across the datasets.

This verification will establish a consistent basis for comparing predictive accuracy, bias, and model complexity.

In [ ]:

# Select Linear Regression prediction columns
linear_results = df_linear_predictions[
    ["Date", "DIVISION", "Intervention_Count", "Linear_Prediction"]
].copy()

# Rename actual intervention counts
linear_results = linear_results.rename(
    columns={"Intervention_Count": "Linear_Actual"}
)

# Check duplicate observations
print(
    "Linear Regression duplicates:",
    linear_results.duplicated(
        subset=["Date", "DIVISION"]
    ).sum()
)

print(
    "Previous model duplicates:",
    df_previous_predictions.duplicated(
        subset=["Date", "DIVISION"]
    ).sum()
)

# Match the four forecasting methods
df_four_models = pd.merge(
    df_previous_predictions,
    linear_results,
    on=["Date", "DIVISION"],
    how="inner",
    validate="one_to_one"
)

# Verify that actual intervention counts match
actual_counts_match = (
    df_four_models["Actual_Count"] ==
    df_four_models["Linear_Actual"]
).all()

# Display verification results
print("\nPrevious model observations:", len(df_previous_predictions))
print("Linear Regression observations:", len(linear_results))
print("Matched observations:", len(df_four_models))
print("Actual intervention counts match:", actual_counts_match)

# Display the four model predictions
display(
    df_four_models[
        [
            "Date",
            "DIVISION",
            "Actual_Count",
            "Baseline_Prediction",
            "Linear_Prediction",
            "LightGBM_Prediction",
            "CatBoost_Prediction"
        ]
    ].head()
)


Linear Regression duplicates: 0
Previous model duplicates: 0

Previous model observations: 8760
Linear Regression observations: 8760
Matched observations: 8760
Actual intervention counts match: True


,Date,DIVISION,Actual_Count,Baseline_Prediction,Linear_Prediction,LightGBM_Prediction,CatBoost_Prediction
0,2021-01-01,1,18.0,22.384615,26.262695,23.121406,25.721262
1,2021-01-01,2,41.0,51.230769,52.819793,54.634131,54.572590
2,2021-01-01,3,48.0,52.923077,51.549891,58.526979,53.516553
3,2021-01-01,4,53.0,58.461538,56.753947,58.386332,56.256852
4,2021-01-01,5,42.0,53.769231,55.034881,58.798730,56.173112


## 6.9. Compare Forecasting Model Performance

We will compare the four forecasting methods using the same 8,760 division-day validation observations from 2021 through 2024.

The comparison will include Mean Absolute Error (MAE), prediction bias, absolute bias, and percentage improvement in MAE relative to the historical baseline.

These metrics will help evaluate whether the additional complexity of LightGBM and CatBoost produces meaningful improvements compared with Linear Regression.

Model selection will also consider performance across operational divisions and validation years before reaching a conclusion.

In [ ]:

# Identify the prediction column for each model
model_columns = {
    "Baseline": "Baseline_Prediction",
    "Linear Regression": "Linear_Prediction",
    "LightGBM": "LightGBM_Prediction",
    "CatBoost": "CatBoost_Prediction"
}

# Store model evaluation results
model_results = []

# Calculate metrics for each model
for model_name, prediction_column in model_columns.items():

    errors = (
        df_four_models[prediction_column] -
        df_four_models["Actual_Count"]
    )

    model_results.append({
        "Model": model_name,
        "MAE": errors.abs().mean(),
        "Bias": errors.mean(),
        "Absolute_Bias": abs(errors.mean())
    })

# Create the comparison table
df_model_comparison = pd.DataFrame(model_results)

# Retrieve the baseline MAE
baseline_mae = df_model_comparison.loc[
    df_model_comparison["Model"] == "Baseline",
    "MAE"
].iloc[0]

# Calculate percentage improvement relative to baseline
df_model_comparison["MAE_Improvement_Percent"] = (
    (baseline_mae - df_model_comparison["MAE"]) /
    baseline_mae * 100
)

# Display the results, sorted by MAE
print("Four-Model Validation Comparison (2021-2024)")

display(
    df_model_comparison.sort_values("MAE").round(3)
)


Four-Model Validation Comparison (2021-2024)


,Model,MAE,Bias,Absolute_Bias,MAE_Improvement_Percent
3,CatBoost,8.127,-1.768,1.768,4.164
2,LightGBM,8.150,-1.913,1.913,3.894
1,Linear Regression,8.384,-1.340,1.340,1.132
0,Baseline,8.480,-0.483,0.483,0.000


**Methodological Observation**

The four forecasting methods were compared using the same 8,760 division-day validation observations from 2021 through 2024.

CatBoost achieved the lowest overall MAE (8.127), followed by LightGBM (8.150), Linear Regression (8.384), and the historical baseline (8.480).

Linear Regression improved MAE by 1.132% relative to the baseline, compared with 3.894% for LightGBM and 4.164% for CatBoost.

However, the gradient boosting models exhibited greater negative prediction bias than Linear Regression and the baseline.

These results show that increased model complexity produced modest improvements in average absolute forecasting accuracy. Further analysis of annual and division-level performance is necessary before selecting the preferred model.

## 6.10. Compare Performance by Division and Year

We will compare the four forecasting methods separately by operational division and validation year.

The analysis will calculate Mean Absolute Error (MAE) for each model using the same matched observations.

We will identify the model with the lowest MAE in each division and year to assess whether the overall performance differences are consistent.

These results will help determine whether the additional complexity of gradient boosting models provides a meaningful and consistent benefit over Linear Regression and the historical baseline.

In [ ]:

# Store absolute prediction errors for each model
for model_name, prediction_column in model_columns.items():

    df_four_models[f"{model_name}_Absolute_Error"] = (
        df_four_models[prediction_column] -
        df_four_models["Actual_Count"]
    ).abs()

# Define the error columns
error_columns = [
    f"{model_name}_Absolute_Error"
    for model_name in model_columns
]

# Calculate MAE by operational division
division_comparison = (
    df_four_models.groupby("DIVISION")[error_columns].mean()
)

# Rename columns
division_comparison.columns = list(model_columns.keys())

# Identify the model with the lowest MAE
division_comparison["Best_Model"] = (
    division_comparison[list(model_columns.keys())].idxmin(axis=1)
)

print("MAE by Division:")
display(division_comparison.round(3))

# Extract the forecast year
df_four_models["Year"] = df_four_models["Date"].dt.year

# Calculate MAE by year
yearly_comparison = (
    df_four_models.groupby("Year")[error_columns].mean()
)

# Rename columns
yearly_comparison.columns = list(model_columns.keys())

# Identify the model with the lowest MAE
yearly_comparison["Best_Model"] = (
    yearly_comparison[list(model_columns.keys())].idxmin(axis=1)
)

print("\nMAE by Year:")
display(yearly_comparison.round(3))


MAE by Division:


,Baseline,Linear Regression,LightGBM,CatBoost,Best_Model
DIVISION,,,,,
1,6.319,6.413,6.035,6.134,LightGBM
2,8.773,8.443,8.255,8.143,CatBoost
3,9.707,9.716,9.442,9.322,CatBoost
4,8.844,8.780,8.679,8.506,CatBoost
5,9.359,9.150,8.846,8.789,CatBoost
6,7.880,7.805,7.644,7.869,LightGBM



MAE by Year:


,Baseline,Linear Regression,LightGBM,CatBoost,Best_Model
Year,,,,,
2021,7.255,7.292,7.191,7.123,CatBoost
2022,7.789,7.568,7.671,7.550,CatBoost
2023,10.719,10.619,9.784,9.958,LightGBM
2024,8.159,8.059,7.954,7.878,CatBoost


**Methodological Observation**

The division-level comparison showed that CatBoost achieved the lowest MAE in four of the six operational divisions, while LightGBM achieved the lowest MAE in the remaining two divisions.

Linear Regression did not achieve the lowest MAE in any division and performed slightly worse than the historical baseline in Divisions 1 and 3.

The annual comparison showed that CatBoost achieved the lowest MAE in 2021, 2022, and 2024, while LightGBM performed best in 2023.

Linear Regression was particularly competitive in 2022 but showed a larger performance gap during 2023.

These results indicate that the gradient boosting models provide measurable forecasting benefits, although their advantages vary across divisions and years. Prediction bias and model complexity must also be considered before selecting the preferred approach.

## 6.11. Compare Prediction Bias by Division and Year

We will compare prediction bias across operational divisions and validation years for all four forecasting methods.

Prediction bias is calculated as the average difference between predicted and actual intervention counts. Positive values indicate overestimation, while negative values indicate underestimation.

This analysis will help identify systematic forecasting differences that may not be visible when comparing MAE alone.

Bias will be considered alongside MAE, because a value close to zero does not necessarily indicate accurate predictions.

In [ ]:

# Calculate signed prediction errors for each model
for model_name, prediction_column in model_columns.items():

    df_four_models[f"{model_name}_Error"] = (
        df_four_models[prediction_column] -
        df_four_models["Actual_Count"]
    )

# Identify the signed error columns
bias_columns = [
    f"{model_name}_Error"
    for model_name in model_columns
]

# Calculate prediction bias by division
division_bias = (
    df_four_models.groupby("DIVISION")[bias_columns].mean()
)

# Use model names as column labels
division_bias.columns = list(model_columns.keys())

print("Prediction Bias by Division:")
display(division_bias.round(3))

# Calculate prediction bias by year
yearly_bias = (
    df_four_models.groupby("Year")[bias_columns].mean()
)

# Use model names as column labels
yearly_bias.columns = list(model_columns.keys())

print("\nPrediction Bias by Year:")
display(yearly_bias.round(3))


Prediction Bias by Division:


,Baseline,Linear Regression,LightGBM,CatBoost
DIVISION,,,,
1,-0.337,-1.079,-1.351,-0.892
2,-0.443,-1.251,-1.496,-0.867
3,-0.597,-1.521,-3.407,-3.806
4,-0.372,-1.200,-2.200,-2.565
5,-0.365,-1.255,-1.798,-1.946
6,-0.783,-1.732,-1.227,-0.530



Prediction Bias by Year:


,Baseline,Linear Regression,LightGBM,CatBoost
Year,,,,
2021,-0.245,-0.928,-0.778,-0.846
2022,-1.187,-0.965,-2.370,-1.945
2023,-0.111,-1.520,-3.530,-3.184
2024,-0.387,-1.945,-0.974,-1.096


## 6.12. Evaluate Prediction Error Variability

We will calculate the standard deviation of prediction errors for the four forecasting methods.

Standard deviation measures how much the signed prediction errors vary around their average value.

This analysis complements MAE and prediction bias by providing information about error variability.

We will calculate the results overall and separately for each operational division. The findings will help inform the provisional model selection without using the reserved final test period.

In [ ]:

# Calculate overall error standard deviation
variability_results = []

for model_name in model_columns:

    error_column = f"{model_name}_Error"

    variability_results.append({
        "Model": model_name,
        "Error_Std": df_four_models[error_column].std()
    })

# Create the overall variability table
df_error_variability = pd.DataFrame(variability_results)

print("Overall Prediction Error Variability:")
display(df_error_variability.round(3))

# Calculate error standard deviation by division
division_variability = (
    df_four_models.groupby("DIVISION")[bias_columns].std()
)

# Rename columns using model names
division_variability.columns = list(model_columns.keys())

print("\nError Standard Deviation by Division:")
display(division_variability.round(3))


Overall Prediction Error Variability:


,Model,Error_Std
0,Baseline,14.228
1,Linear Regression,14.205
2,LightGBM,13.844
3,CatBoost,13.830



Error Standard Deviation by Division:


,Baseline,Linear Regression,LightGBM,CatBoost
DIVISION,,,,
1,14.601,14.693,14.155,14.225
2,13.547,13.332,13.077,12.939
3,17.332,17.498,17.000,16.850
4,13.137,13.142,12.987,12.672
5,15.320,15.157,14.627,14.714
6,10.528,10.432,10.210,10.471


**Methodological Observation**

The error variability analysis showed that CatBoost achieved the lowest overall standard deviation of prediction errors (13.830), followed closely by LightGBM (13.844).

Linear Regression (14.205) and the historical baseline (14.228) showed slightly greater error variability.

At the division level, CatBoost had the lowest error standard deviation in Divisions 2, 3, and 4, while LightGBM performed best in Divisions 1, 5, and 6.

These findings suggest that the gradient boosting models provide modest improvements in both average absolute accuracy and error variability. However, their stronger negative prediction bias remains an important limitation.

The results support selecting a preferred model through a balanced assessment of predictive accuracy, bias, temporal stability, interpretability, and complexity.

## 6.13. Provisional Model Selection

Four forecasting methods were evaluated using the same 48 monthly chronological validation windows from 2021 through 2024: the historical same-weekday baseline, Linear Regression, LightGBM, and CatBoost.

The evaluation considered MAE, prediction bias, error variability, performance across operational divisions, and temporal consistency.

CatBoost achieved the lowest overall MAE (8.127) and error standard deviation (13.830), improving MAE by 4.164% relative to the historical baseline.

Linear Regression offered a simpler and more interpretable alternative but achieved a higher overall MAE (8.384). LightGBM performed similarly to CatBoost, with an MAE of 8.150.

All machine learning models showed greater negative prediction bias than the baseline. CatBoost also demonstrated substantial underestimation in certain divisions, particularly Division 3.

**Provisional Decision:** CatBoost is selected as the preferred single-model approach based on its overall validation performance. However, its advantage over LightGBM is small, and the additional complexity of a multi-model ensemble is not currently justified by the available evidence.

The next step is to evaluate the selected approach on the reserved 2025–2026 test period, without using those results to revise the model selection.

## 6.14. Evaluate Forecasting Errors on High-Demand Days

We will examine forecasting performance on days with unusually high recorded intervention counts.

High demand will be defined separately for each operational division using the 90th percentile of historical daily counts available within each monthly training window.

This approach accounts for differences in typical intervention volumes between divisions and prevents future validation outcomes from influencing the high-demand thresholds.

We will compare MAE and prediction bias across the four forecasting methods for observations classified as high demand.

The analysis is retrospective: actual intervention counts are used to identify high-demand observations for evaluation, not as information available when forecasts are generated.

In [ ]:

# Store high-demand thresholds for all validation months
high_demand_thresholds = []

for _, window in df_schedule.iterrows():

    # Select observations available at the monthly training cutoff
    training_data = df_modeling[
        df_modeling["Date"] <= window["Training_Cutoff"]
    ]

    # Calculate the 90th percentile separately for each division
    division_thresholds = (
        training_data.groupby("DIVISION")["Intervention_Count"]
        .quantile(0.90)
    )

    # Store thresholds for the validation month
    for division, threshold in division_thresholds.items():

        high_demand_thresholds.append({
            "Validation_Month": window["Validation_Start"],
            "DIVISION": division,
            "High_Demand_Threshold": threshold
        })

# Create a threshold dataset
df_thresholds = pd.DataFrame(high_demand_thresholds)

# Display verification results
print("Monthly validation windows:", len(df_schedule))
print("Threshold records:", len(df_thresholds))
print("Missing thresholds:", df_thresholds["High_Demand_Threshold"].isna().sum())

print("\nFirst six thresholds:")
display(df_thresholds.head(6))


Monthly validation windows: 48
Threshold records: 288
Missing thresholds: 0

First six thresholds:


,Validation_Month,DIVISION,High_Demand_Threshold
0,2021-01-01,1,30.0
1,2021-01-01,2,59.0
2,2021-01-01,3,66.0
3,2021-01-01,4,67.0
4,2021-01-01,5,63.0
5,2021-01-01,6,47.0


**Methodological Observation**

High-demand thresholds were calculated using the 90th percentile of historical daily intervention counts for each operational division and monthly validation window.

A total of 288 thresholds were generated, corresponding to 48 validation months and six divisions. No missing thresholds were identified.

Each threshold was calculated using only the historical observations available through the corresponding monthly training cutoff, preventing validation outcomes from influencing the threshold calculation.

The division-specific thresholds account for differences in typical intervention volumes. They will be used to evaluate forecasting errors on historically defined high-demand days.

## 6.15. Identify High-Demand Validation Observations

We will identify high-demand observations within the 2021–2024 validation period.

Each division-day observation will be matched with the 90th-percentile threshold calculated from the historical training data available at the beginning of its validation month.

An observation will be classified as high demand when its actual intervention count is greater than or equal to the corresponding threshold.

Actual intervention counts will be used only for retrospective evaluation. This classification does not represent a model-generated high-demand warning.

In [ ]:

# Create the validation month identifier
df_four_models["Validation_Month"] = (
    df_four_models["Date"].dt.to_period("M").dt.to_timestamp()
)

# Match each observation with its monthly division threshold
df_high_demand = pd.merge(
    df_four_models,
    df_thresholds,
    on=["Validation_Month", "DIVISION"],
    how="left",
    validate="many_to_one"
)

# Identify actual high-demand observations
df_high_demand["Is_High_Demand"] = (
    df_high_demand["Actual_Count"] >=
    df_high_demand["High_Demand_Threshold"]
)

# Verify the classification
print("Total validation observations:", len(df_high_demand))

print(
    "Missing thresholds:",
    df_high_demand["High_Demand_Threshold"].isna().sum()
)

print(
    "High-demand observations:",
    df_high_demand["Is_High_Demand"].sum()
)

print(
    "High-demand percentage:",
    round(df_high_demand["Is_High_Demand"].mean() * 100, 2)
)

# Count high-demand observations by division
print("\nHigh-demand observations by division:")
display(
    df_high_demand.groupby("DIVISION")["Is_High_Demand"].sum()
)


Total validation observations: 8760
Missing thresholds: 0
High-demand observations: 1916
High-demand percentage: 21.87

High-demand observations by division:


DIVISION
1    366
2    306
3    322
4    271
5    286
6    365
Name: Is_High_Demand, dtype: int64

**Methodological Observation**

The high-demand classification identified 1,916 observations out of 8,760 validation records, representing 21.87% of the division-day observations.

All validation observations were successfully matched with their corresponding historical division-specific thresholds, with no missing thresholds.

The proportion of high-demand observations exceeded 10%. This is possible because the 90th-percentile thresholds were calculated from earlier training periods rather than from the validation observations themselves.

The results indicate that high-demand occurrence should be examined using the historical thresholds without assuming that exactly 10% of future observations will exceed them.

This classification is used for retrospective error analysis and does not establish whether the models can identify high-demand days in advance.

## 6.16. Compare Forecasting Performance on High-Demand Days

We will compare the four forecasting methods using the validation observations classified as high demand.

For each method, we will calculate Mean Absolute Error (MAE) and prediction bias within the high-demand subset.

We will also examine the results by operational division to identify differences in forecasting performance.

Because high-demand observations are identified using actual intervention counts, this analysis measures retrospective forecasting errors on high-demand days. It does not measure the models' ability to issue high-demand warnings before those days occur.

The results will complement the overall validation metrics and help assess whether the selected model performs adequately during periods of elevated intervention workload.

In [ ]:

# Select observations with actual high demand
df_high_demand_only = df_high_demand[
    df_high_demand["Is_High_Demand"]
].copy()

# Store performance results
high_demand_results = []

# Evaluate each forecasting method
for model_name, prediction_column in model_columns.items():

    # Calculate prediction errors
    errors = (
        df_high_demand_only[prediction_column] -
        df_high_demand_only["Actual_Count"]
    )

    # Store MAE and Bias
    high_demand_results.append({
        "Model": model_name,
        "MAE": errors.abs().mean(),
        "Bias": errors.mean()
    })

# Create the comparison table
df_high_demand_metrics = pd.DataFrame(high_demand_results)

print("High-Demand Model Comparison")
print("Observations:", len(df_high_demand_only))

display(
    df_high_demand_metrics.sort_values("MAE").round(3)
)

# Calculate MAE by division
division_high_demand_mae = pd.DataFrame()

for model_name, prediction_column in model_columns.items():

    # Calculate absolute errors
    df_high_demand_only[f"{model_name}_Absolute_Error"] = (
        df_high_demand_only[prediction_column] -
        df_high_demand_only["Actual_Count"]
    ).abs()

    # Calculate MAE for each division
    division_high_demand_mae[model_name] = (
        df_high_demand_only.groupby("DIVISION")[
            f"{model_name}_Absolute_Error"
        ].mean()
    )

print("\nHigh-Demand MAE by Division:")
display(division_high_demand_mae.round(3))


High-Demand Model Comparison
Observations: 1916


,Model,MAE,Bias
0,Baseline,15.366,-15.107
1,Linear Regression,16.026,-14.544
2,LightGBM,16.073,-15.417
3,CatBoost,16.078,-15.799



High-Demand MAE by Division:


,Baseline,Linear Regression,LightGBM,CatBoost
DIVISION,,,,
1,11.441,12.374,11.963,11.607
2,16.239,16.419,16.427,15.954
3,18.085,19.902,21.065,21.616
4,17.204,17.293,18.564,19.088
5,18.311,18.123,18.439,18.967
6,12.497,13.356,11.788,11.281


**Methodological Observation**

The high-demand evaluation included 1,916 division-day observations identified using historical 90th-percentile thresholds.

The historical baseline achieved the lowest MAE (15.366) on these observations, followed by Linear Regression (16.026), LightGBM (16.073), and CatBoost (16.078).

All four methods showed substantial negative prediction bias on high-demand observations, indicating that actual elevated workloads were generally underpredicted.

Performance varied by division, and no single model achieved the lowest high-demand MAE across all six divisions.

Because these observations were selected using actual intervention counts, the analysis evaluates retrospective forecasting errors rather than the ability to identify high-demand days in advance. A separate evaluation of predicted high-demand warnings is needed before drawing conclusions about their potential planning usefulness.

## 6.17. Evaluate High-Demand Warning Detection

We will evaluate whether the four forecasting methods could identify high-demand observations using their predicted intervention counts.

A high-demand warning will be issued when a model's predicted count meets or exceeds the historical 90th-percentile threshold for the corresponding division and validation month.

We will calculate recall, precision, and the number of warnings for each method.

Recall measures the proportion of actual high-demand observations correctly identified. Precision measures the proportion of issued warnings that correspond to actual high demand.

This analysis evaluates a proposed warning rule retrospectively. The threshold is an analytical definition of elevated demand, not an established SIM operational alert threshold.

In [ ]:

# Store warning detection results
warning_results = []

# Count actual high-demand observations
actual_high_demand = df_high_demand["Is_High_Demand"]
total_high_demand = actual_high_demand.sum()

for model_name, prediction_column in model_columns.items():

    # Identify predictions that exceed the historical threshold
    predicted_high_demand = (
        df_high_demand[prediction_column] >=
        df_high_demand["High_Demand_Threshold"]
    )

    # Count correctly detected high-demand observations
    correct_warnings = (
        predicted_high_demand & actual_high_demand
    ).sum()

    # Count all warnings issued
    total_warnings = predicted_high_demand.sum()

    # Calculate recall and precision
    recall = correct_warnings / total_high_demand

    if total_warnings > 0:
        precision = correct_warnings / total_warnings
    else:
        precision = np.nan

    warning_results.append({
        "Model": model_name,
        "Warnings": total_warnings,
        "Correct_Warnings": correct_warnings,
        "Recall": recall,
        "Precision": precision
    })

# Create the comparison table
df_warning_metrics = pd.DataFrame(warning_results)

print("Actual high-demand observations:", total_high_demand)
print("\nHigh-Demand Warning Detection:")
display(df_warning_metrics.round(3))



Actual high-demand observations: 1916

High-Demand Warning Detection:


,Model,Warnings,Correct_Warnings,Recall,Precision
0,Baseline,539,213,0.111,0.395
1,Linear Regression,471,198,0.103,0.420
2,LightGBM,297,152,0.079,0.512
3,CatBoost,359,174,0.091,0.485


**Methodological Observation**

The high-demand warning evaluation included 1,916 actual high-demand division-day observations identified using historical 90th-percentile thresholds.

The historical baseline achieved the highest recall (11.1%), correctly identifying 213 high-demand observations. Linear Regression achieved 10.3% recall, CatBoost 9.1%, and LightGBM 7.9%.

LightGBM achieved the highest precision (51.2%), followed by CatBoost (48.5%), Linear Regression (42.0%), and the historical baseline (39.5%).

These results reveal a trade-off between the number of warnings issued, the proportion of correct warnings, and the proportion of high-demand observations detected.

All four methods showed low recall under the proposed warning rule. Therefore, the current results do not support claiming that the models can reliably identify unusually high-demand days in advance.

CatBoost remains the provisional preferred model for forecasting daily intervention counts based on overall MAE. However, its suitability for high-demand warning detection has not been established.

The findings suggest that improving high-demand identification would require further investigation of the warning thresholds, model predictions, and relevant evaluation criteria.

## 6.18. Final Validation Conclusion and Provisional Model Selection

Four forecasting methods were evaluated using 48 monthly chronological validation windows covering 8,760 division-day observations from 2021 through 2024.

CatBoost achieved the lowest overall MAE (8.127), representing a 4.164% improvement over the historical same-weekday baseline (8.480). LightGBM achieved a similar MAE (8.150), while Linear Regression achieved 8.384.

CatBoost also had the lowest overall standard deviation of prediction errors (13.830). However, all three machine learning models showed greater negative prediction bias than the historical baseline, and CatBoost did not achieve the lowest MAE in every division or year.

Additional evaluation of 1,916 high-demand observations revealed an important limitation. The historical baseline achieved the lowest MAE on this subset (15.366), while CatBoost achieved 16.078. All four methods showed substantial underestimation on high-demand observations.

When applying historical 90th-percentile thresholds as a proposed warning rule, recall ranged from 7.9% to 11.1% across the four methods. These results do not support reliable advance identification of high-demand observations using the current warning approach.

**Provisional Model Selection**

CatBoost is selected as the preferred single-model approach for forecasting next-day daily intervention counts because it achieved the lowest overall validation MAE.

LightGBM remains a competitive alternative, with only a small difference in overall MAE. Linear Regression provides a simpler comparison model but does not achieve equivalent accuracy across the validation periods and divisions.

A multi-model ensemble will not be developed at this stage because the current results do not establish a need for additional model complexity.

**Limitations and Next Steps**

The selected model should not be presented as an established high-demand warning system. Further work would be required to improve high-demand identification and determine whether the forecasts could support practical resource-planning decisions.

The reserved 2025–2026 test period remains unevaluated and will be used for a final out-of-sample assessment of the selected forecasting approach and historical baseline.